<a href="https://colab.research.google.com/github/jacielefreitas63-tech/projeto-delivery-logistic/blob/main/1_limpeza_e_eda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧪 Projeto Delivery: Engenharia de Dados & Análise Exploratória (EDA)


# 1 Configuração do Ambiente e Conexão com o Data Lake (Google Drive)

Neste notebook, realizaremos a ingestão, limpeza fina, tratamento de outliers e união das bases transacionais da *Olist, malha rodoviária do **OpenStreetMap* e dados meteorológicos do *INMET*.

In [1]:
!pip install osmnx networkx pandas numpy pyspark

In [2]:
# Importação das bibliotecas fundamentais para manipulação e análise estatística
import pandas as pd
import numpy as np
import networkx as nx
import os as ox

# Imports específicos do PySpark para processamento distribuído
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
print("🚀 Bibliotecas carregadas com sucesso!")

🚀 Bibliotecas carregadas com sucesso!


In [3]:
# Inicialização do motor do Spark
spark = SparkSession.builder \
    .appName("LogisticaDelivery") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

print("🚀 Spark inicializado com sucesso e pronto para processamento distribuído!")

🚀 Spark inicializado com sucesso e pronto para processamento distribuído!


In [4]:
# Conexão segura com o armazenamento de dados do Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 📦2.Ingestão da Base de Pedidos (Olist Orders)
> O objetivo desta etapa é ler o dataset transacional de ordens de serviço, inspecionar os tipos de dados primitivos e mapear o volume inicial de registros antes de aplicar as regras de limpeza.

In [5]:
# Caminho
base_path = '/content/drive/MyDrive/projeto_delivery_logistica/data/'

In [6]:
import pandas as pd
import gc
import os

# Mapeamento dos datasets originais
arquivos = {
    'customers': 'olist_customers_dataset.csv',
    'items': 'olist_order_items_dataset.csv',
    'orders': 'olist_orders_dataset.csv',
    'geolocation': 'olist_geolocation_dataset.csv',
    'reviews': 'olist_order_reviews_dataset.csv'
}


In [7]:
# Processamento individual (não salva em dicionário!)
for nome, arquivo in arquivos.items():
    print(f"Processando {nome}...")
    caminho_csv = os.path.join(base_path, arquivo)
    caminho_parquet = os.path.join(base_path, f"{nome}.parquet")

    # 1. Carrega o CSV
    df = pd.read_csv(caminho_csv)

    # 2. Otimização simples de tipos (economia de RAM)
    for col in df.select_dtypes(include=['object']).columns:
        if df[col].nunique() < 1000:
            df[col] = df[col].astype('category')

    # 3. Salva em Parquet
    df.to_parquet(caminho_parquet, index=False)

    # 4. LIMPEZA TOTAL DA RAM
    del df
    gc.collect()
    print(f"✅ {nome} salvo e memória limpa.")

Processando customers...
✅ customers salvo e memória limpa.
Processando items...
✅ items salvo e memória limpa.
Processando orders...
✅ orders salvo e memória limpa.
Processando geolocation...
✅ geolocation salvo e memória limpa.
Processando reviews...
✅ reviews salvo e memória limpa.


Tratamento Individual (Limpeza pré-merge)
Tratamos o CEP aqui para garantir que o merge funcione perfeitamente.

In [8]:
# Tratamento do CEP (Não use df_raw!)
# Carregue apenas os dois necessários para o tratamento de CEP
df_cust = pd.read_parquet(os.path.join(base_path, 'customers.parquet'))
df_geo = pd.read_parquet(os.path.join(base_path, 'geolocation.parquet'))

df_cust['customer_zip_code_prefix'] = df_cust['customer_zip_code_prefix'].astype(str).str.zfill(5)
df_geo['geolocation_zip_code_prefix'] = df_geo['geolocation_zip_code_prefix'].astype(str).str.zfill(5)

# Salva de volta
df_cust.to_parquet(os.path.join(base_path, 'customers.parquet'))
df_geo.to_parquet(os.path.join(base_path, 'geolocation.parquet'))

del df_cust, df_geo
gc.collect()
print("✅ CEPs tratados e salvos.")

✅ CEPs tratados e salvos.


In [9]:
# --- Conversão e Preparação para Spark (Otimizada) ---

# Carregamos os dados diretamente via Spark do formato Parquet.
# Isso evita o gargalo de memória ao converter Pandas -> Spark.
# Os caminhos abaixo assumem que você salvou os arquivos no formato 'nome.parquet' anteriormente.

df_pedidos = spark.read.parquet(os.path.join(base_path, 'orders.parquet'))
df_clientes = spark.read.parquet(os.path.join(base_path, 'customers.parquet'))
df_reviews = spark.read.parquet(os.path.join(base_path, 'reviews.parquet'))

# Para a geolocalização, lemos a versão já limpa/única que você criou anteriormente
# Certifique-se de que o nome do arquivo corresponde ao que foi salvo
df_geolocation = spark.read.parquet(os.path.join(base_path, 'geolocation.parquet')) \
    .withColumnRenamed("geolocation_zip_code_prefix", "customer_zip_code_prefix") \
    .withColumnRenamed("geolocation_lat", "customer_lat") \
    .withColumnRenamed("geolocation_lng", "customer_lng") \
    .dropDuplicates(subset=["customer_zip_code_prefix"])

print("✅ Bases de dados carregadas e convertidas para Spark com sucesso.")

✅ Bases de dados carregadas e convertidas para Spark com sucesso.


 Consolidação (Merge) das Bases

In [10]:
from pyspark.sql import functions as F

# Carregamento e preparação da base de itens para agregação
df_items = spark.read.parquet(os.path.join(base_path, 'items.parquet'))

# Cálculo do frete total por pedido (agrupamento e agregação)
df_freight = df_items.groupBy("order_id").agg(F.sum("freight_value").alias("valor_frete"))

# Consolidação do modelo principal (Star Schema) via left joins para preservação da integridade dos registros
df_olist_consolidado = df_pedidos \
    .join(df_clientes, "customer_id", "left") \
    .join(df_reviews, "order_id", "left") \
    .join(df_geolocation, "customer_zip_code_prefix", "left") \
    .join(df_freight, "order_id", "left")

In [ ]:
df_olist_consolidado.printSchema()

Limpeza Final, Validação e Exportação
Finalizamos removendo nulos e validando os dados antes de salvar.

In [11]:
from pyspark.sql import functions as F
from pyspark.sql.types import DateType

# --- Etapa: Limpeza e Tratamento Final ---

# 1. Tratamento de nulos e estruturação de geolocalização
df_final = df_olist_consolidado \
    .fillna({'review_score': -1}) \
    .withColumn("tem_geolocalizacao", F.when(F.col("customer_lat").isNotNull(), True).otherwise(False)) \
    .fillna({'customer_lat': 0.0, 'customer_lng': 0.0})

In [12]:
# 2. Padronização temporal e criação de variáveis de negócio
# Convertemos datas com tratamento de strings vazias ou NaN para evitar falhas no cast
df_final = df_final \
    .withColumn("dt_entrega_real", F.to_date(F.when(F.col("order_delivered_customer_date").isin(["", "NaN"]), None)
                                           .otherwise(F.col("order_delivered_customer_date")))) \
    .withColumn("dt_entrega_estimada", F.to_date(F.col("order_estimated_delivery_date"))) \
    .withColumn("dias_atraso", F.datediff(F.col("dt_entrega_real"), F.col("dt_entrega_estimada"))) \
    .withColumn("is_gargalo", F.when(F.col("dias_atraso") > 0, 1).otherwise(0)) \
    .withColumnRenamed("customer_state", "estado")

In [13]:
# 3. Auditoria de Qualidade e Exportação
total_registros = df_final.count()
print(f"--- Auditoria de Qualidade: {total_registros} registros processados ---")

--- Auditoria de Qualidade: 99992 registros processados ---


In [14]:
# Persistência do modelo tratado em formato Parquet para garantir eficiência de armazenamento
caminho_saida = os.path.join(base_path, 'df_completo_final.parquet')
df_final.write.mode("overwrite").parquet(caminho_saida)

print("✅ Pipeline de tratamento concluída com sucesso e dados exportados.")

✅ Pipeline de tratamento concluída com sucesso e dados exportados.


## 🌤️3.Ingestão e Tratamento de Dados Meteorológicos (INMET)
> O objetivo desta etapa é carregar o histórico climático, tratar valores nulos de precipitação (chuva) e preparar a base para cruzamento temporal com os momentos de compra e entrega.

In [15]:
# Caminho completo e direto
BASE_PATH = '/content/drive/MyDrive/projeto_delivery_logistica/data/'

In [16]:
from pyspark.sql import functions as F

# 1. Carregamento Otimizado (evita inferência de esquema lenta)
df_inmet = spark.read.option("header", "true").csv(BASE_PATH + 'southeast.csv')

Conversão de Tipos e Normalização
Esta etapa é crucial para garantir a integridade dos dados antes da limpeza. Convertendo as colunas numéricas e de data agora, evitamos erros de sintaxe e inconsistências durante os cálculos matemáticos e o cruzamento de informações (JOIN) com outras bases

In [18]:
from pyspark.sql import functions as F
import re

# Helper function to clean column names for Spark
def clean_spark_column_name(col_name):
    # Replace accented characters with their non-accented counterparts
    name = col_name.replace('Á', 'A').replace('À', 'A').replace('Â', 'A').replace('Ã', 'A') \
                   .replace('É', 'E').replace('È', 'E').replace('Ê', 'E') \
                   .replace('Í', 'I').replace('Ì', 'I').replace('Î', 'I') \
                   .replace('Ó', 'O').replace('Ò', 'O').replace('Ô', 'O').replace('Õ', 'O') \
                   .replace('Ú', 'U').replace('Ù', 'U').replace('Û', 'U') \
                   .replace('Ç', 'C')
    # Replace special characters and spaces with underscore
    name = re.sub(r'[^a-zA-Z0-9_]', '_', name)
    # Replace multiple underscores with a single one
    name = re.sub(r'_{2,}', '_', name)
    # Remove leading/trailing underscores
    name = name.strip('_')
    # Convert to lowercase as per desired convention
    return name.lower()

# 1. Limpeza de colunas
# Primeiro, renomeia todas as colunas para nomes compatíveis com o Spark
for c in df_inmet.columns:
    new_col_name = clean_spark_column_name(c)
    df_inmet = df_inmet.withColumnRenamed(c, new_col_name)

# Agora, todos os nomes das colunas em df_inmet estão limpos. Podemos referenciá-los diretamente.
# Define os nomes das colunas limpas para referência consistente
cleaned_precipitation_col = clean_spark_column_name("PRECIPITAÇÃO TOTAL, HORÁRIO (mm)")
cleaned_data_col = clean_spark_column_name("Data")

# 2. Convertendo tipos e tratando valores nulos/inválidos (-9999.0)
df_inmet = df_inmet.withColumn(cleaned_precipitation_col,
                                F.when(F.col(cleaned_precipitation_col) == -9999.0, None)
                                 .otherwise(F.col(cleaned_precipitation_col).cast("double"))) \
                   .withColumn(cleaned_data_col, F.to_date(F.col(cleaned_data_col)))

In [19]:
# 3. Join Otimizado com Trim nas chaves
# Carrega df_station, que não estava definida
df_station = spark.read.parquet(BASE_PATH + 'stations_tratado.parquet')

# Renomeia a coluna 'state' de df_station para 'station_state' para evitar ambiguidade após o join
df_station_clean = df_station.withColumn("station_code", F.trim(F.col("station_code"))) \
                             .withColumnRenamed("state", "station_state")

df_inmet_clean = df_inmet.withColumn("station_code", F.trim(F.col("station_code")))

df_joined = df_inmet_clean.join(df_station_clean, "station_code", "left")

In [20]:
# 4. Agregação Final (Médias e Máximos)
df_final_inmet = df_joined.groupBy("station_state", "data") \
    .agg(
        F.avg("precipitacao_total_horario_mm").alias("media_chuva_mm"),
        F.max("precipitacao_total_horario_mm").alias("max_chuva_mm")
    )

In [21]:
# 5. Exportação
df_final_inmet.write.mode("overwrite").parquet(BASE_PATH + 'inmet_preparado.parquet')
print("✅ Pipeline INMET finalizada com sucesso!")

✅ Pipeline INMET finalizada com sucesso!


## 🗺️4.Ingestão e Tratamento de Dados de Malha Rodoviária (OpenStreetMap)
> Nesta etapa, mapeamos as coordenadas geográficas (latitude e longitude) dos clientes e vendedores para calcular as distâncias reais das rotas e entender o impacto do tráfego e das rodovias no tempo de entrega.

In [22]:
# ---
# Pipeline de Processamento e Tratamento: Malha Rodoviária
# Objetivo: Otimizar a carga, limpar inconsistências e preparar dados para modelagem geoespacial.
# Responsável pela eficiência: Otimização de I/O, tipagem de dados e particionamento estratégico.
# ---

from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType

# 1. Ingestão Otimizada (Lazy Evaluation)
# Selecionamos apenas as colunas essenciais ('projection pushdown') para reduzir drasticamente o consumo de memória RAM.
df_malha_rodoviaria = spark.read.parquet(BASE_PATH + 'malha_rodoviaria_particionada/') \
    .select("u", "v", "length", "highway", "speed_kph", "travel_time", "estado") \
    .dropna(subset=["u", "v"]) \
    .dropDuplicates()

In [23]:
# 2. Refinamento de Dados e Tipagem
# Garantimos a integridade do schema (casting) para evitar falhas de runtime em cálculos matemáticos ou joins futuros.
df_malha_tratada = df_malha_rodoviaria \
    .withColumn("speed_kph", F.col("speed_kph").cast(DoubleType())) \
    .withColumn("length", F.col("length").cast(DoubleType()))

In [24]:
# 3. Persistência de Dados (Output Estratégico)
# Utilizamos 'partitionBy' para organizar os dados fisicamente no disco por estado.
# Isso acelera consultas futuras, permitindo que o Spark leia apenas a partição necessária.
# O codec 'snappy' oferece o melhor equilíbrio entre velocidade de compressão e I/O.
df_malha_tratada.write.mode("overwrite") \
    .partitionBy("estado") \
    .option("compression", "snappy") \
    .parquet(BASE_PATH + 'malha_rodoviaria_tratada.parquet')

print("✅ Pipeline de Malha Rodoviária finalizada com sucesso.")

✅ Pipeline de Malha Rodoviária finalizada com sucesso.


In [25]:
# Rode isso para ter certeza absoluta:
df_malha_rodoviaria.printSchema()
df_malha_rodoviaria.select("highway").distinct().show()

root
 |-- u: long (nullable = true)
 |-- v: long (nullable = true)
 |-- length: double (nullable = true)
 |-- highway: string (nullable = true)
 |-- speed_kph: double (nullable = true)
 |-- travel_time: double (nullable = true)
 |-- estado: string (nullable = true)

+--------------------+
|             highway|
+--------------------+
|['trunk', 'motorw...|
|['trunk_link', 't...|
|['primary', 'resi...|
|['primary', 'moto...|
|          trunk_link|
|['unclassified', ...|
|['unclassified', ...|
|['unclassified', ...|
|['trunk', 'primary']|
|        primary_link|
|            tertiary|
|['unclassified', ...|
|            motorway|
|['living_street',...|
|['primary', 'moto...|
|         residential|
|       emergency_bay|
|['trunk', 'motorw...|
|               trunk|
|['tertiary', 'res...|
+--------------------+
only showing top 20 rows


In [26]:
# Lista as colunas do seu DataFrame atual
print(df_malha_tratada.columns)

['u', 'v', 'length', 'highway', 'speed_kph', 'travel_time', 'estado']


## Integração de Dados Logísticos (Data Mart)

In [27]:
from pyspark.sql import functions as F

# 1. Carregamento das bases tratadas
# Mantemos o uso de caminhos padronizados para consistência do projeto
df_pedidos = spark.read.parquet(base_path + 'df_completo_final.parquet')
df_inmet = spark.read.parquet(base_path + 'inmet_preparado.parquet')
df_malha = spark.read.parquet(base_path + 'malha_rodoviaria_tratada.parquet')

In [28]:
# 2. Padronização de chaves para Join
# Ajuste necessário para garantir que as colunas de união tenham os mesmos nomes
df_inmet_renamed = df_inmet.withColumnRenamed("station_state", "estado")

# Verificando o schema de df_pedidos antes de qualquer modificação nesta célula para depuração.
df_pedidos.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_zip_code_prefix: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: string (nullable = true)
 |-- order_approved_at: string (nullable = true)
 |-- order_delivered_carrier_date: string (nullable = true)
 |-- order_delivered_customer_date: string (nullable = true)
 |-- order_estimated_delivery_date: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- estado: string (nullable = true)
 |-- review_id: string (nullable = true)
 |-- review_score: long (nullable = true)
 |-- review_comment_title: string (nullable = true)
 |-- review_comment_message: string (nullable = true)
 |-- review_creation_date: string (nullable = true)
 |-- review_answer_timestamp: string (nullable = true)
 |-- customer_lat: double (nullable = true)
 |-- customer_lng: double (nullable = true)
 |-- geo

In [29]:
# 3. Join Otimizado (Denormalização)
# Unimos as bases mantendo a integridade dos pedidos (Left Join)

# Assegurando que df_pedidos tenha as colunas 'estado' e 'data' para o join
df_pedidos = df_pedidos.withColumnRenamed("customer_state", "estado")
df_pedidos = df_pedidos.withColumn("data", F.to_date(F.col("order_purchase_timestamp")))

# Verificando o schema de df_pedidos antes do join para depuração.
df_pedidos.printSchema()

df_integrado = df_pedidos.join(df_inmet_renamed, on=['data', 'estado'], how='left') \
                         .join(df_malha, on=['estado'], how='left')

root
 |-- order_id: string (nullable = true)
 |-- customer_zip_code_prefix: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: string (nullable = true)
 |-- order_approved_at: string (nullable = true)
 |-- order_delivered_carrier_date: string (nullable = true)
 |-- order_delivered_customer_date: string (nullable = true)
 |-- order_estimated_delivery_date: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- estado: string (nullable = true)
 |-- review_id: string (nullable = true)
 |-- review_score: long (nullable = true)
 |-- review_comment_title: string (nullable = true)
 |-- review_comment_message: string (nullable = true)
 |-- review_creation_date: string (nullable = true)
 |-- review_answer_timestamp: string (nullable = true)
 |-- customer_lat: double (nullable = true)
 |-- customer_lng: double (nullable = true)
 |-- geo

In [30]:
# 4. Seleção e Renomeação para o Data Mart
df_final = df_integrado.select(
    F.col("order_id").alias("id_pedido"),
    F.col("estado"),
    F.col("order_purchase_timestamp"),
    F.col("order_delivered_customer_date"),
    F.col("dias_atraso").alias("tempo_entrega"),
    F.col("media_chuva_mm").alias("precipitacao"),
    F.col("speed_kph").alias("velocidade_kmh"),
    F.col("travel_time")
)

In [ ]:
# 5 e 6. Otimização e Escrita do Data Mart Final
# Mantemos o df_final na memória (RAM) em vez de escrever no disco
df_final.cache()

# Agora, apenas escrevemos o arquivo final diretamente
caminho_final = base_path + 'data_mart_looker_final.parquet'
df_final.write.mode("overwrite").parquet(caminho_final)

print("✅ Data Mart (Gold) finalizado e pronto para exportação!")

## 🚚 5. Engenharia de Atributos (Feature Engineering): Métricas de SLA, e Análise de Correlação.
Nesta etapa, criamos as métricas de performance para avaliar o desempenho logístico e validamos, atráves de uma análise estatística, se variáveis como pluviosidade, transito...

In [ ]:
from pyspark.sql import functions as F
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

# Carregando o Data Mart consolidado que geramos na etapa anterior
# Isso substitui a necessidade de refazer joins aqui
df_analise = spark.read.parquet(base_path + 'data_mart_looker_final.parquet')

print("✅ Data Mart carregado com sucesso para análise!")
df_analise.show(5)

In [ ]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.stat import Correlation

# Selecionamos apenas as colunas numéricas que definimos para a análise
colunas_correlacao = ['precipitacao', 'velocidade_kmh', 'tempo_entrega']

# Agrupamento para processamento Spark ML
assembler = VectorAssembler(inputCols=colunas_correlacao, outputCol="features_vector", handleInvalid="skip")
df_vetor = assembler.transform(df_analise)

# Cálculo da matriz de correlação (Pearson)
matriz_spark = Correlation.corr(df_vetor, "features_vector").head()[0]

In [ ]:
# Conversão para formato Pandas para plotagem
matriz_dados = matriz_spark.toArray()
df_matriz_heatmap = pd.DataFrame(matriz_dados, columns=colunas_correlacao, index=colunas_correlacao)

# Plotagem
plt.figure(figsize=(10, 8))
sns.heatmap(df_matriz_heatmap, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title("Diagnóstico: Correlação entre Clima, Trânsito e Tempo de Entrega")
plt.show()

#🚀 6. Análise Preditiva de Atraso Logístico (Machine Learning)
Nesta etapa final do projeto, utilizamos Machine Learning para transformar dados históricos em inteligência preditiva. Nosso objetivo é treinar um modelo capaz de antecipar a probabilidade de um pedido sofrer atraso na entrega.

In [ ]:
# 1. Imports essenciais para o fluxo de ML
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.sql.functions import when, col

# 2. Preparação dos dados (Pipeline direto)
# Criamos o 'target' e vetorizamos as features sem ler arquivos CSV extras
df_ml = df_final.withColumn("target", when(col("atraso_entrega_dias") > 0, 1).otherwise(0))

feature_cols = ['media_chuva_mm', 'tempo_entrega_dias', 'geolocation_lat', 'geolocation_lng']
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
df_model = assembler.transform(df_ml)

In [ ]:
# 3. Divisão e Treinamento
train_data, test_data = df_model.randomSplit([0.8, 0.2], seed=42)

rf = RandomForestClassifier(labelCol="target", featuresCol="features", numTrees=100)
model = rf.fit(train_data)

In [ ]:
# 4. Avaliação de Performance
predictions = model.transform(test_data)
evaluator = BinaryClassificationEvaluator(labelCol="target", rawPredictionCol="rawPrediction")
auc = evaluator.evaluate(predictions)

print(f"Desempenho do Modelo (AUC): {auc:.2f}")

In [ ]:
# Extrair importância das features
importances = model.featureImportances
feature_names = ['media_chuva_mm', 'tempo_entrega_dias', 'geolocation_lat', 'geolocation_lng']

# Mapear e ordenar
feature_importance_list = list(zip(feature_names, importances))
sorted_importances = sorted(feature_importance_list, key=lambda x: x[1], reverse=True)

print("Importância das Variáveis:")
for name, importance in sorted_importances:
    print(f"{name}: {importance:.4f}")

In [ ]:
# Exibir exemplos reais de previsão
predictions.select("target", "prediction", "probability").show(10)